In [2]:
import os
import json
import warnings
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
DATA_PATH = "./training_data_normalized.csv"
OUT_BASE = "./dag_out/GES"
RANDOM_STATE = 42
MAX_FEATURES = None

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

# GES score config (요구사항: 문자열로 전달)
GES_SCORE_FUNC = "local_score_BIC"

# cycle break: |score| 가장 작은 edge 제거
CYCLE_BREAK_STRATEGY = "min_abs_score"


# =========================
# Data
# =========================
def load_numeric_X(
    data_path: str,
    drop_target_candidates: bool = True,
    max_features: Optional[int] = None,
    random_state: int = 42
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    np.random.seed(random_state)
    df = pd.read_csv(data_path, low_memory=False)

    if drop_target_candidates:
        drop_cols = [c for c in df.columns if c in TARGET_CANDIDATES]
        if drop_cols:
            print(f"[INFO] drop target candidates: {drop_cols}")
            df = df.drop(columns=drop_cols)

    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    if max_features is not None and df.shape[1] > max_features:
        df = df.iloc[:, :max_features].copy()
        print(f"[INFO] feature capped: {max_features}")

    # 결측치: median
    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    X = df.values.astype(float)

    # (중요) score 계산의 일관성을 위해 센터링
    X = X - X.mean(axis=0, keepdims=True)

    print(f"[INFO] X shape: {X.shape}")
    return df, X, col_names


# =========================
# Graph helpers
# =========================
def extract_directed_edges_from_causallearn_graph(G: Any, d: int) -> List[Tuple[int, int]]:
    """
    causal-learn Graph 객체에서 i->j 방향 간선을 추출.
    버전/표현차 대응:
      - PDAG/CPDAG는 방향이 일부만 확정될 수 있음(여기서는 "확정된 i->j"만 추출)
      - mat 값 패턴이 다양할 수 있어 몇 가지 케이스를 순차적으로 시도
    """
    mat = getattr(G, "graph", None)
    if mat is None:
        raise ValueError("Cannot find adjacency matrix 'G.graph' in causal-learn Graph object.")

    mat = np.asarray(mat)
    if mat.shape[0] != d or mat.shape[1] != d:
        raise ValueError(f"G.graph shape mismatch: {mat.shape} vs d={d}")

    edges: List[Tuple[int, int]] = []

    # Case A (가장 흔함): i->j : mat[i,j]==1 and mat[j,i]==-1
    for i in range(d):
        for j in range(d):
            if i == j:
                continue
            if mat[i, j] == 1 and mat[j, i] == -1:
                edges.append((i, j))

    if edges:
        return list(dict.fromkeys(edges))

    # Case B: 반대 부호 케이스: i->j : mat[i,j]==-1 and mat[j,i]==1
    for i in range(d):
        for j in range(d):
            if i == j:
                continue
            if mat[i, j] == -1 and mat[j, i] == 1:
                edges.append((i, j))

    if edges:
        return list(dict.fromkeys(edges))

    # Case C: 일부 구현에서 방향만 표시되고 반대편은 0일 수 있음
    # mat[i,j]!=0 and mat[j,i]==0 을 i->j로 해석 (최후 수단)
    for i in range(d):
        for j in range(d):
            if i == j:
                continue
            if mat[i, j] != 0 and mat[j, i] == 0:
                edges.append((i, j))

    # 중복 제거
    edges = list(dict.fromkeys(edges))
    return edges


def break_cycles_by_removing_small_edges(W: np.ndarray) -> np.ndarray:
    """
    cycle이 있으면, cycle에 포함된 edge 중 |W|가 가장 작은 edge 제거.
    """
    W2 = W.copy()

    def build_graph(Wm):
        G = {i: [] for i in range(Wm.shape[0])}
        for i in range(Wm.shape[0]):
            for j in range(Wm.shape[1]):
                if i != j and abs(Wm[i, j]) > 0:
                    G[i].append(j)
        return G

    def find_cycle_edges(adj):
        d = len(adj)
        color = [0] * d  # 0=unvisited,1=visiting,2=done
        parent = [-1] * d

        def dfs(u):
            color[u] = 1
            for v in adj[u]:
                if color[v] == 0:
                    parent[v] = u
                    cyc = dfs(v)
                    if cyc is not None:
                        return cyc
                elif color[v] == 1:
                    # back edge 발견 -> cycle 복원
                    nodes = [v]
                    cur = u
                    while cur != v and cur != -1:
                        nodes.append(cur)
                        cur = parent[cur]
                    nodes.append(v)
                    nodes = nodes[::-1]
                    return [(a, b) for a, b in zip(nodes[:-1], nodes[1:])]
            color[u] = 2
            return None

        for s in range(d):
            if color[s] == 0:
                cyc = dfs(s)
                if cyc is not None:
                    return cyc
        return None

    while True:
        cyc = find_cycle_edges(build_graph(W2))
        if cyc is None:
            break

        mags = [(abs(W2[i, j]), i, j) for (i, j) in cyc]
        mags.sort(key=lambda x: x[0])
        _, i_min, j_min = mags[0]
        W2[i_min, j_min] = 0.0

    return W2


# =========================
# BIC score wrapper (버전 호환)
# =========================
def make_local_bic_scorer(X: np.ndarray):
    """
    causal-learn 버전에 따라 local_score_BIC가
      - (X, i, PAi, ...) 형태의 함수일 수도 있고
      - local_score_BIC(X) -> callable(i, PAi) 형태의 팩토리일 수도 있음
    둘 다 지원하는 래퍼를 반환한다: scorer(i, PAi) -> float
    """
    try:
        from causallearn.score.LocalScoreFunction import local_score_BIC
    except Exception as e:
        raise ImportError(
            "local_score_BIC를 찾을 수 없습니다. causal-learn 설치가 필요합니다.\n"
            "pip install causal-learn\n"
            f"Original error: {e}"
        )

    # 1) 팩토리 형태인지 먼저 시도: local_score_BIC(X) 가 callable 반환
    try:
        maybe = local_score_BIC(X)
        if callable(maybe):
            def scorer(i: int, PAi: List[int]) -> float:
                return float(maybe(i, PAi))
            return scorer
    except TypeError:
        pass
    except Exception:
        # 팩토리 시도 중 다른 예외는 무시하고 다음 케이스로
        pass

    # 2) 함수 형태: local_score_BIC(X, i, PAi, ...) 직접 호출
    def scorer(i: int, PAi: List[int]) -> float:
        return float(local_score_BIC(X, i, PAi))
    return scorer


# =========================
# GES Δ local BIC score
# =========================
def compute_delta_scores_for_edges(
    X: np.ndarray,
    edges: List[Tuple[int, int]],
    d: int
) -> np.ndarray:
    """
    Δscore(i->j) = local_score(j | Pa_j ∪ {i}) - local_score(j | Pa_j)
    여기서 Pa_j는 최종 그래프에서의 부모 집합에서 i를 제외한 것.
    결과: W_score (d x d) where W[i,j]=Δscore if i->j else 0
    """
    scorer = make_local_bic_scorer(X)

    parents_of: Dict[int, List[int]] = {j: [] for j in range(d)}
    for i, j in edges:
        parents_of[j].append(i)

    W = np.zeros((d, d), dtype=float)

    for i, j in edges:
        pa_wo_i = sorted([p for p in parents_of[j] if p != i])

        s0 = scorer(j, pa_wo_i)
        s1 = scorer(j, sorted(pa_wo_i + [i]))

        W[i, j] = float(s1 - s0)

    return W


# =========================
# Save artifacts
# =========================
def save_artifacts(W: np.ndarray, col_names: List[str], out_dir: str, alg_name: str) -> None:
    import networkx as nx

    os.makedirs(out_dir, exist_ok=True)

    edges = []
    for i, src in enumerate(col_names):
        for j, tgt in enumerate(col_names):
            if i != j and abs(W[i, j]) > 0:
                edges.append([src, tgt, float(W[i, j])])

    # edges_<ALG>.csv : score 컬럼
    edge_df = pd.DataFrame(edges, columns=["source", "target", "score"])
    edge_path = os.path.join(out_dir, f"edges_{alg_name}.csv")
    edge_df.to_csv(edge_path, index=False)

    # adj_<ALG>.csv : Δscore 인접행렬
    adj_df = pd.DataFrame(W, index=col_names, columns=col_names)
    adj_path = os.path.join(out_dir, f"adj_{alg_name}.csv")
    adj_df.to_csv(adj_path)

    # graph exports
    G = nx.DiGraph()
    for n in col_names:
        G.add_node(n)
    for _, r in edge_df.iterrows():
        G.add_edge(r["source"], r["target"], score=float(r["score"]))

    graphml_path = os.path.join(out_dir, f"graph_{alg_name}.graphml")
    gexf_path = os.path.join(out_dir, f"graph_{alg_name}.gexf")
    nx.write_graphml(G, graphml_path)
    nx.write_gexf(G, gexf_path)

    nodes = [{"id": n} for n in G.nodes()]
    jedges = [{"source": u, "target": v, "score": float(G[u][v].get("score", 0.0))} for u, v in G.edges()]
    json_path = os.path.join(out_dir, f"graph_{alg_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"nodes": nodes, "edges": jedges}, f, ensure_ascii=False, indent=2)

    print(f"[SAVE] {alg_name}")
    print(f"  - {edge_path} (n_edges={len(edge_df)})")
    print(f"  - {adj_path}")
    print(f"  - {graphml_path}")
    print(f"  - {gexf_path}")
    print(f"  - {json_path}")


# =========================
# Main
# =========================
def main():
    _, X, col_names = load_numeric_X(
        DATA_PATH,
        drop_target_candidates=True,
        max_features=MAX_FEATURES,
        random_state=RANDOM_STATE
    )

    try:
        from causallearn.search.ScoreBased.GES import ges
    except Exception as e:
        raise ImportError(
            "GES 실행을 위해 causal-learn이 필요합니다.\n"
            "pip install causal-learn\n"
            f"Original error: {e}"
        )

    # 1) Run GES (score_func는 문자열로 전달)
    Record = ges(X, score_func=GES_SCORE_FUNC)

    # 2) Get Graph object (버전 차 대응)
    if isinstance(Record, dict) and "G" in Record:
        G = Record["G"]
    else:
        G = getattr(Record, "G", None)
        if G is None:
            raise ValueError("Cannot access learned graph from GES result. Expected Record['G'] or Record.G")

    d = len(col_names)

    # 3) Extract directed edges from learned graph
    edges_ij = extract_directed_edges_from_causallearn_graph(G, d)
    print(f"[INFO] directed edges from GES: {len(edges_ij)}")

    # 4) Compute Δ local BIC score per edge
    W_score = compute_delta_scores_for_edges(X, edges_ij, d)

    # 5) Enforce DAG by breaking cycles (remove smallest |score|)
    if CYCLE_BREAK_STRATEGY == "min_abs_score":
        W_score = break_cycles_by_removing_small_edges(W_score)
    else:
        raise ValueError(f"Unknown CYCLE_BREAK_STRATEGY: {CYCLE_BREAK_STRATEGY}")

    # 6) Save artifacts
    save_artifacts(W_score, col_names, OUT_BASE, "GES")


if __name__ == "__main__":
    main()


[INFO] drop target candidates: ['label']
[INFO] X shape: (17881, 14)
[INFO] directed edges from GES: 55
[SAVE] GES
  - ./dag_out/GES\edges_GES.csv (n_edges=55)
  - ./dag_out/GES\adj_GES.csv
  - ./dag_out/GES\graph_GES.graphml
  - ./dag_out/GES\graph_GES.gexf
  - ./dag_out/GES\graph_GES.json
